In [0]:
-- ============================================================
-- AEGIS SUPPLY CHAIN INTELLIGENCE PLATFORM
-- SILVER LAYER VERIFICATION
--
-- Purpose:
--   Confirm that all Silver tables were created successfully
--   and that their row counts match their Bronze sources.
-- ============================================================

SELECT 'customer' AS table_name, COUNT(*) AS row_count
FROM workspace.aegis_silver.customer

UNION ALL
SELECT 'product', COUNT(*)
FROM workspace.aegis_silver.product

UNION ALL
SELECT 'supplier', COUNT(*)
FROM workspace.aegis_silver.supplier

UNION ALL
SELECT 'supplier_product', COUNT(*)
FROM workspace.aegis_silver.supplier_product

UNION ALL
SELECT 'warehouse', COUNT(*)
FROM workspace.aegis_silver.warehouse

UNION ALL
SELECT 'carrier', COUNT(*)
FROM workspace.aegis_silver.carrier

UNION ALL
SELECT 'order', COUNT(*)
FROM workspace.aegis_silver.order

UNION ALL
SELECT 'order_item', COUNT(*)
FROM workspace.aegis_silver.order_item

UNION ALL
SELECT 'shipment', COUNT(*)
FROM workspace.aegis_silver.shipment

UNION ALL
SELECT 'shipment_item', COUNT(*)
FROM workspace.aegis_silver.shipment_item

UNION ALL
SELECT 'inventory', COUNT(*)
FROM workspace.aegis_silver.inventory

UNION ALL
SELECT 'return', COUNT(*)
FROM workspace.aegis_silver.return

ORDER BY table_name;

-- ============================================================
-- SILVER DERIVED-FIELD VERIFICATION
-- ============================================================

-- 1. ORDER ITEM: revenue calculation
SELECT
    COUNT(*) AS total_rows,
    SUM(
        CASE
            WHEN revenue = ROUND(quantity * unit_price, 2)
            THEN 0
            ELSE 1
        END
    ) AS revenue_errors
FROM workspace.aegis_silver.order_item;


-- 2. SHIPMENT: delivery metrics
SELECT
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN is_late = TRUE THEN 1 ELSE 0 END) AS late_shipments,
    SUM(CASE WHEN is_late = FALSE THEN 1 ELSE 0 END) AS on_time_shipments,
    SUM(CASE WHEN actual_delivery_date IS NULL THEN 1 ELSE 0 END) AS undelivered
FROM workspace.aegis_silver.shipment;


-- 3. INVENTORY: available quantity + stockout
SELECT
    COUNT(*) AS total_rows,
    SUM(
        CASE
            WHEN available_quantity = stock_quantity - reserved_quantity
            THEN 0
            ELSE 1
        END
    ) AS available_quantity_errors,
    SUM(
        CASE
            WHEN stockout_flag =
                 (available_quantity <= 0)
            THEN 0
            ELSE 1
        END
    ) AS stockout_flag_errors
FROM workspace.aegis_silver.inventory;